## Parse
Parse movie data from https://grouplens.org/datasets/movielens/ ml-latest-small.
This is used to create data from Collaborative filtering.

In [ ]:
import re
import csv
import pandas as pd
import numpy as np
from numpy.random import default_rng
from collections import defaultdict
from numpy import loadtxt
num_features = 10

In [ ]:
# user_dict: from ratings.csv
#            user_dict[user_id]
#                              [movie_id] = rating
# movie_dict: from movies.csv
#            movie_dict[movie_id] = string movie description
# movie_stats:
#            movie_stats[movie_id]
#                                 ["rating_ar"] = np.array() of ratings

In [ ]:
# count on these to maintain order in python 3.7 or greater
user_dict = defaultdict(dict)  # user_id: {movie_id:rating}  for movies in assigment
movie_dict = defaultdict(str)  # movie_id : string to display

def loadMovieCSV( movie_dict, min_year = 2000):
    """ reads movies.csv from ml-latest-small, 
    returns list of names of columns and the data in a two dim list
    """
    p = re.compile('^(.+)\s\((\d+)\)')
    count = 0
    with open('./movies.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: # skip header
                count +=1
            else:
                count +=1
                movie_id = int(line[0])   
                if movie_id in [30892, 85565]: print(f"movies without rating eliminated {movie_id}"); continue
                m = p.match(line[1])      # crack title
                if m == None: 
                    #print(line[1]); 
                    continue  # if it doesn't match filter, leave it out (there are about 10 of these)
                title = m.group(1)
                year = int(m.group(2))
                if year >= min_year:
                    if movie_id == 3384: print(f"3384  {line[1]}")
                    if len(line[1]) < 5: print(f"unexpected length of title {len(line[1])}")
                    movie_dict[movie_id] = line[1]
        return 

In [ ]:
loadMovieCSV(movie_dict, min_year = 2000 )
num_movies = len(movie_dict.keys())
print(num_movies)

In [ ]:
def loadUserRatings( movie_dict, user_dict):
    """ read ratings.csv and put into user_dict
        only put in movies in movie_dict 
        ratings.csv in user_id, then movie_id order
    """
    count = 0
    with open('./ratings.csv', newline='') as csvfile:
        reader = csv.reader(csvfile, delimiter=',', quotechar='"')
        for line in reader:
            if count == 0: 
                count += 1 
                #print(line)  # skip header
            else:
                count += 1
                #if count > 5: break
                user_id = int(line[0])
                movie_id = int(line[1]) 
                rating = line[2]
                if rating not in ['0.5', '1.0', '1.5', '2.0', '2.5', '3.0', '3.5', '4.0', '4.5', '5.0']: print(f" unexpected rating {rating}")
                if movie_id not in movie_dict: continue
                user_dict[user_id][movie_id] = float(rating)
        return 

In [ ]:
loadUserRatings(movie_dict, user_dict)
num_users = len(user_dict.keys())
print(num_users)

In [ ]:
def make_YR(user_dict, movie_dict,  num_movies, num_users):
    """ since we pared down movies and users, movie_id and user_id are not the same as the index. """
    Y = np.zeros((num_movies,num_users))
    R = np.zeros((num_movies,num_users), dtype=np.int32)
    ulist = list(user_dict.keys())  # for indexing
    mlist = list(movie_dict.keys()) 
    chksum = 0
    for user_id in user_dict.keys():
        for movie_id in user_dict[user_id].keys():
            user_idx = ulist.index(user_id)
            item_idx = mlist.index(movie_id)
            rating  = user_dict[user_id][movie_id]
            Y[item_idx,user_idx] = float(rating)
            chksum += float(rating)
            R[item_idx,user_idx] = 1
    pd.DataFrame(np.array(Y)).to_csv("./parsed/small_movies_Y.csv",header=None, index=False)
    pd.DataFrame(np.array(R)).to_csv("./parsed/small_movies_R.csv",header=None, index=False)
    return(Y,R, chksum)

In [ ]:
Y, R, check = make_YR(user_dict, movie_dict, num_movies, num_users)
print(Y.shape, R.shape, check)

In [ ]:
# do a bit of checking
for i in range(len(Y)):
    sumrow = sum(Y[i,:])
    smtrue = sum(Y[i,:] > 0)
    sumRow = sum(R[i,:] )
    assert sumrow >= sumRow/2, f"{i}, sum of Rs  {sumrow:0.2f} should be less than or equal to ratings {sumRow:0.2f} "
    assert smtrue == sumRow,   f"{i}, number of ratings {smtrue}  should be equal to sum of Rs {sumRow} "
    
asum = 0
for user_id in user_dict.keys():
    for movie_id in user_dict[user_id].keys():
        asum += float(user_dict[user_id][movie_id])
assert check == asum, f"{check} should equal {asum}"
assert np.sum(Y * R) == check, f"all ratings should match"


In [ ]:
def make_X(num_movies,num_features):
    """ make an X matrix for developing cost. I guess the reaon for doing this random load here and storing in a file
        is so that different version of numpy or other libraries won't change the results
    """
    rng = default_rng(123)
    X = rng.standard_normal(num_movies * num_features, dtype=np.float32).reshape((num_movies,num_features))
    pd.DataFrame(np.array(X)).to_csv("./parsed/small_movies_X.csv",header=None, index=False)
    return(X)    

In [ ]:
X=make_X(num_movies,num_features)
print(X.shape, X[:1])

In [ ]:
def make_Wb():
    rng = default_rng(1234)
    W = rng.random((num_users,num_features), dtype=np.float32) - 0.5
    b = rng.random((1,num_users), dtype=np.float32) - 0.5
    pd.DataFrame(np.array(W)).to_csv("./parsed/small_movies_W.csv", header=None, index=False)
    pd.DataFrame(np.array(b)).to_csv("./parsed/small_movies_b.csv", header=None, index=False)
    return(W, b)

In [ ]:
W, b = make_Wb()

## Don't forget to run make moviestats below. Have to get stats first.

# generate some stats to help pick good movies

In [ ]:
# offsets into array
MOVIE_ID = 0; AVE = 1; VAR=2; MAX=3; MIN=4; Number=5;

def gen_movie_stats(user_dict, movie_dict,  num_movies, num_users):
    """ move movie stats into an array - could be a good use of numpy structured arrays to keep track of columns """
    movie_stats_dict = defaultdict(lambda: defaultdict(list))
    mlist = list(movie_dict.keys()) 

    # pull per user stats into per movie stats
    for user_id in user_dict.keys():
        for movie_id in user_dict[user_id].keys():
            movie_stats_dict[movie_id]['rlist'].append(user_dict[user_id][movie_id]) # rating
            
    num_movies = len(movie_stats_dict.keys())
    df = pd.DataFrame(index=range(num_movies), columns=['idx', 'movie id', 'mean', 'var', 'max', 'min', 'num'])
    for i, movie_id in enumerate(movie_stats_dict.keys()):
        ar = np.array(movie_stats_dict[movie_id]['rlist'])
        df.loc[i,'idx']  = mlist.index(movie_id)  # row in Y,R
        df.loc[i,'mean'] = np.mean(ar)
        df.loc[i,'var']  = np.var(ar)
        df.loc[i,'max']  = np.amax(ar)
        df.loc[i,'min']  = np.amin(ar)
        df.loc[i,'num']  = len(ar)
        df.loc[i,'movie id'] = movie_id    
        df.loc[i,'title'] = movie_dict[movie_id]
    df.sort_values(["idx"], inplace=True )
    df.reset_index(drop=True, inplace=True)
    return (df)


In [ ]:
movie_stats_df = gen_movie_stats(user_dict, movie_dict,  num_movies, num_users)

In [ ]:
df=movie_stats_df[movie_stats_df["num"]>20].sort_values(["mean","num"], ascending=False)
df[:20]

In [ ]:
df=movie_stats_df.sort_values(["num"], ascending=False)
df[:10]

In [ ]:

movie_stats_df[movie_stats_df['title'].str.contains( "Pirates")]

### make movielist with movie_stats

In [ ]:
movie_stats_df[["mean","num", "title"]].to_csv("./parsed/small_movie_list.csv", header=["mean rating", "number of ratings","title"], index=True)

In [ ]:
#testing
#def load_Movie_List_pd():
#    """ returns df with and index of movies in the order they are in in the Y matrix """
#    df = pd.read_csv('./parsed/small_movie_list.csv', header=0, index_col=0,  delimiter=',', quotechar='"')
#    return(df)


In [ ]:
#movie_df = load_Movie_List_pd()
#movie_df.loc[[1,4,6]].sort_values("mean rating", ascending=False)

In [ ]:
#def make_movielist(movie_stats):
#    movielist = []
#    for movie_id in movie_dict.keys():
#        movielist.append([movie_id, movie_dict[movie_id]])
#    pd.DataFrame(movielist).to_csv("./parsed/small_movie_list.csv",header=['original movie_id', 'title'], index=True)

In [ ]:
#make_movielist()

In [ ]:
movie_stats_df

## find the two movies which are missing ratings
only needed to do once

In [ ]:
filt = movie_stats_df["mean"]==0
movie_status_df.loc[filt]

In [ ]:
mstat_set = set( movie_status_df["movie id"].to_list())
len(mstat_set)

In [ ]:
mdict_set = set(movie_dict.keys())
len(mdict_set)

In [ ]:
mstat_set ^ mdict_set

{30892, 85565}